# Sensibilidad de las métricas v4 a la relectura guiada por modelos
**Sin GPU, sin re-entrenar.** Usa las predicciones fuera de fold del benchmark v4 (7 clasificadores) y cambia solo las etiquetas de referencia.

- **Escenario 1 — v4:** etiquetas v4 tal cual (debe reproducir la Table 3: Swin-T macro-F1 0,838).
- **Escenario 2 — originales:** en los recortes releídos que están en v4, `micdas` ← `micdas_original` (etiqueta antes de la relectura).
- **Escenario 3 — sin releídos:** se excluyen los recortes releídos.

Métricas: macro-F1 (IC 95 % bootstrap por niño, 2000), κ cuadrático, Se/Sp B+C (ICDAS ≥ 3), Se/Sp cualquier caries (ICDAS ≥ 1), recall C. Diferencias pareadas (mismos niños remuestreados en todos los modelos y escenarios). Swin-T vs cada modelo con Holm, por escenario.

Lectura: el escenario 2 vuelve a poner errores conocidos (cota baja), el 1 puede ser optimista (cota alta); el 3 es el más limpio pero con menos datos y menos C.

**Inputs:** `benchmark-icdas-v4` (`oof_predicciones.csv`), `base-icdas-v4` (`manifest.csv` o `icdas_v4.zip`). `relectura_etiquetas.csv` va incrustado en la celda 2; si se agrega como dataset, se usa ese.

In [ ]:
import os, glob, io, gzip, base64, hashlib, zipfile, re, numpy as np, pandas as pd
IN, W = '/kaggle/input', '/kaggle/working'
CLASSES = ['0', 'A', 'B', 'C']; CI = {c: i for i, c in enumerate(CLASSES)}
NAMES = {'swin_tiny': 'Swin-T', 'convnextv2_tiny': 'ConvNeXt V2-T', 'resnet50': 'ResNet50', 'coatnet_0': 'CoAtNet-0',
         'maxvit_tiny': 'MaxViT-T', 'vit_small': 'ViT-S/16', 'yolo11m_cls': 'YOLO11m-cls'}
REF = 'swin_tiny'; NB = 2000; SEED = 0
find = lambda n: (glob.glob(f'{IN}/**/{n}', recursive=True) or [None])[0]
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

In [ ]:
# ---- relectura_etiquetas.csv: dataset si existe; si no, copia incrustada (verificada por hash) ----
REL_B64 = "H4sIADBR6WoC/6Vd245dN459n28pDCSSuj0mBuYDBvMeVLuqezxwXA07+f+RyLMlnnN02dr1kqRil6BNUeQiF0l9e3v5+vPj3398y//+32/f38p//PPj+9vLx89v/3r/8fKvn3//++Pl29e3119/lP/17cfr95c/H36WH3++f3//+tffP19f3t6/fvv17ePH7Q9e3/7v77dvX1/fPo7fVP/n6+uf//j28Yf8wfHnf368vX//+PXHr7Lk17zSf/y3Mcbe1jM2+vjyxTgKL/Tyj9df7y+/3vM++Nd+vr69vsDLby/4n+alban89PvL//z8+/3ld14MjsXyai4vZhGuL4Z1sYhUFvPpBbuL2byYzb/O//jt5b9ev+e/Y3gRqp+XwJbPc+Z+R//8+/3nX++8hLnbT/nJyH5kKXcsBcH6vJQPmD+Dl/r68eOvnx/f8/Z+5++4fYxsRL7G16/xKZSvcWHwNWURd7eV8tMX2coXXizkE/7xlneSqOzEJZ9/68+sNm8f8LhcPocXuluOHiUdb8sRYFEBhKIC98vdxNT9wN94kXRbxDjIuzWAiz2Vj7p9mZKTNfXAgucDo5EKrY/MNuV2rI8uK1J/sfKLcLcYsC7xYvyBtio3QMSyWID7E1Sq1NdGi20/pgjJQRpq42CJptBG5Jwg//W9W2Gd+hQ7+ZTxcecbcDtuH1lnDCmdOe5DVs4XX37fq3NmFbah3ocU2FSEkd2ZbCI24wXEyuK630HjJVKTZ0x5Cco3lTblCVVpszjYVEE1Ddt6BrYtZi0vZgYnfGIxOI4pWT6mZNWtPI6pex35mPJ3yO/nTbB3yFaZRre6SLnYlpuB0cvQsY1ginwwf5i5sIyrdo/V1uNimfI91aE+L1dtMlDk65hweB2LZG/i1Sd/WGIbqWhPdlX5bw92NFZkOCwwFhuRP4zieBla+geotjiw2UO2oecvJ1Z9Bs+f5bMp7uvzejPY9Dmy37S+er6p32TJYLW61rIGU3Z+ZstUIVat8UUa3pTL2XVvJ76m2V+X7NSljO0FaijB8k0PV/z8fg47TASOxYMj3z12uhjaR6F8VB8lrQ0ONpOcbXLZkDUTCPh0q2Q/VX8zIik+LhsNc/XEqCEKZORF2cLuekxqShwSuyvvB4uskS0pJCFuKyNbc38jpquwpKmCCUDiVUIaXNITW2pqLTfeZZ2ASziZmnJbwzjJpa5yT6TdwDIywLU0coHja09BxQ9uGj/A0L7TYZhvHoIy2KHrhpka2ggu9dDtTTbjHblmmw2jdef8tmycPdyWSWxUvVEm8dDB9fe4FvAlhoHg4lBt+mftmho7xgbeukmYNxAKtejKS3TlttGka+jCcXxn7GeiKtcsM8O4YnzMBvRyTX9Jgmm6cNDVHBdbwxE5DRHy4qyb7nrke519D2zuxzftDaFcqJDd+Z5x8M0Uo+OzphE4mSzSYEVMwBcxbsIK3xTXAkfhfIk290FN+12RhwfflUexsfbufOyD5/XN7iLjHBfN7ic1uwsSx+QjNgPzNHcovkGKIuwCTNxmtOwblEghsKXD7m7WqMQfgALBjwIHhZLmFzs0HbYCAf3+xQwqNQFuArnG8gnN+JIkyLIXH1/Ip5BK9tG0GKx8TByY37WYgzLDnN/I33ZRm4NTUQPcRw2nb1fw6kpg70psQKRwhHpZ2GzPs/yffeaRvPvIP91JW9aocOJ26EDjAHaM10OqyziqRv0yKokqfcFwoNgz2IqD4wEocoIBBoj9RIAfj3QFeTEayT6FaicuV2yp42LBygfF43KtU7WxaXFgu2O921a9eIAJV8B9Hzgqi4P3KtO0LjaTbAOfDdpJ3nggjwYlyh0t8qDJF83vQWyQwkP3kp/ZUWrXm/2vDXiZKUgtwrvB2ZbDO477sDg1GyRhizqx1HAFoEAlN8m7zc1XOhQZo2C35MeKvL6hqSl0klgmuK63GEs8Ka02ApHTwBZOFmkYw5CdBh/jq5GaVQ6MSX3CwU5OSKZSIZGjB+fCyLNPPquptHMsXmceqZ1y5kVnOoqTlCsPPVe+Foo1KuUWOAWYjc5FV2WNUmWHExy3dOm2UXpQlu1E82u1KWmA220ohjh/W8S5d1jsSYHmyCEjxt1oz5rDQGcDVESEcZLZnixzRHnZSYR6IXY9ujWhXSwXqx5vflIDzAYlGUg9NR6aLtHEps6RGDTRxZDE5tTf7XIiewoy5kk4s8vJ+1Ekng+ClOOCxGNP+rSZpsmOXY4nezWFZxuTl/NdfMNy2gumHpU39QicrD2YEfS2qHJJMJt9NWyMXqZNzSSxNLZAldEjw1YMCD4BKq0i+IDXG2UecIQGraL3DAOxpxjyjGRSI3k4SipM8AM46AqFf7sxe9m2471tPw0trKL0gpCdfjMYtpXIs17CNMKdbFIhJKsYbtmBeGi/zjk/wXsRQwMPjpOOltLQocxvTuXwHDLVgP4zVJdtHJ69fdc0uz9NXVtoOotCvWL/M8tS7byfDyuqdXxvnROIAFSumPjLaBv5WzSNHqIBPbQhbcXlBWLs72HieReLNYPsJfc1ZBnHjhNV/oJ9zFP+4rS3QpW94AIP6/z2qSlqDyU9P2RlqmMY6hE25S6p9gtJNNuoPS6OGN/ctc9T1B7Ibiw9ErlDlFxuQDVASdLr8TLAbZxeBjqxlwNeCqYxejkKwc9FIVYzeoy8XDAXpUwKWUjGfpjVWysQqaDPSxaNrnKEBeEctSSGqwqihrqPpFG37ELWaVodrOTeR1m59aWlWoCRgQ/nWdJOgYulZqqNADmAnlLPHTzV/HIM7DeS3fLOitSTwjBv+oVhY0PoVEJZcM6Ew5/L1CmkbHk39jLZbRW/h85Oc7Dre+Z0EgPvkxjnT8s5VcMRl2WBC2G1bIaXWg4H2266cX0Zg1g2rW6XS8oljsf9NDeMNoy2TkhaoQ+IU/SxXszXONBzDU85x09gPm9b6G7YEMGooGcs8sYAZg4ksWkcAccTO2olRoIbaS/QlUWOUPDQSrIXKmht4wE5cu94/I3b633N63PgUjAWbJYX2cYFZgLETMKfE/tRFZ9GQDHtURWlCqRm1EHYju3KU9s4wIyszZJ7GyxS9dgHJk1SGB/4ZBmoNSpMdhDbxMu3K9RkHVoul3NpxFycWKwZbArzIob14TdOMHcXwH1mSofy/RhWkYEgDjKYCbxfbEWptDBMfruorPA5TZlNT5nPnL4y1Dezsb9IVFX3saUXtopwbLSKbYVJAcL6pCMoG4afilrioc5UUHq+ZDF8JgKOqqGE68VHRQk0FnYLER0X7nlwk8K0xed51b+Tpv07ayyjeEI0fpH/6Ot0PEBIpj1jTRBsOzHND5o4pf/XHEbjB3P6LU0J3fEVafQgWKYHnbcXVbKSg85xhPdJg52wJW4tJ24/gfsaTQiYJF2K22RWowmzHgn/FB9h+jAtLtJWHCGHZQXuj9FZP3+cGqki0YIP5irJnG97lTKn9HpS3jm0pPrBzLQfbChnMKbyahw1+OTme+oCT1B0oeea7afE1+raQ6MJs5eet20t+T0wqk2KwdUFQhUUSegZKZa65g5YoJEOQmv4Yyv2hbOpF5MnYFQpEic7R85xqTpgVOsUsW0MfrKvgXhiIxxvfmyzABIUUYjcOkq2G4x384FgVQrPums5clDcYJCCKrqcXIBGEFqTXK8V7GwGBRQ9GG/dbbBL54Jq9SP2xuPGhckirnYWSXU1uQsVZ2A1zDA9mPFAAKkrpcQSWuDuOXC/UiUGNrYIJawjlMEiqpz5VoDp91LI0MjBHFC4Xq7/LCQAUOiCz3pcZjHZj0LNUUp87OadBmV640N70/pOg+oiYUfyRNWrY7EjLakMYQ6IGa0Z+wl4BIohjOJNyO+LVsd7NI33JsKNlcOVZkrCK3exEYMgreTex92ID1qLXwZH3RKhM4so8yulGYQXnZrq8wOu6XnKgJ4tWYLa72cFFd2XXi6T8SU5cGQrucZNooVreBo0Hyi9+kM+cO1WUBWDCkPZikGXda2AoeWJcBB2nC2Ch0YGZrVm/BAuM0uAKodhzdSNr3emuEG4BbG4mLRQqYInDadW18zlXcUR7pfnACloIdo97DJfS0sxhIlDhwxwt22RYgbBu+n9X+tla/mzxpl+4ZldNTTmupRD1hTT9WhG9f15DJVt2LP3ihaUuSElk4m7MlYQQ+7ItHJtLuNGEebYUdoqL3ZygaIKkQS4x4tmWzUAksQitE1cgVPl+iynT4yNAaeKQznh52HWoN1NZYJiCL00jlxthwXNDyapcgmXDWXjCcGy63c+bAdNTpluqVnNbvKyWiaVD3W9fOiJHbWuwOxrH8S9Rp1eVXKwcQ3DypLJFtScgdTtWz+N5VtjYHbSaQk45+rjlZ0G7JVRzpu8+JY2StCWoo27SH0TXFVC0MgQpBKB2asgwh+IBKU2yfDBbQNir5pcDfQ8/jLuUaSgjHVQodOyShSC0t9Ik7hnrH9BkSfJTPOu469oHYE55y0Rxn4UqToCHddWeTRXKTto/B9YgGmT7InFWkIuyTkH2q7VgcYDZkPRzbyuCpNBsX/AXcyfAU+NBczWmLPU6LaznaFN3+JUZXHMOzFPNC1mSk9lVKczM40CNFKOdaFbDRT5J60ZqjdMV5n1oxzV+2fooaBr+2wU2xeFDaE4iZLnplPxflLD7d0F4dR+k8ClNJJGvhicKr4vJD+danJCVjUrB8z8eecG5YFPuiMHpyg/6Wp9hCVLI5pMzRzxtTSz9tFZ2yUotg9l2FvcHPZW5rJVp+tsD2WfOO7W+JeJ+YvsPqjWPwEkpXxlF6Sr1j8K1G8gmHJ6oDg9lHBo6uK6/epQ+/0yYuCWEIzbswqhcnm5hzsMpnadBjGp1X6KcK2/wHhj4/IMp9I851MmmYapsUE119Px1f5EjSOquZ4ydPBpJN1ZZgQbt5dhm5+GQ8Pzw8btcWvKrIRriSqwNQLK2Jdpi8RSHbARfVays27Iay3RNqq2QKmD+USpCDbKL/fZS2Iz7o7mQkX6WY6MyWxPZcXaF2gNT06hTqvZrIzmCy+isIbMgRnSs2NNUpyfF94a99XRYh1XxqCncEuwbwFsyzsL/cgR6C4Gw8r4HdX1AS+UkWIb8CmdPgh0OQuOtg2sjRJS0E6VPlqVspBLFdzQAM2vVOP9GC/c1absBfqomgOJydULk+WwMX95EXuN5UXF/GGUNBxMWIa5BQNVzIzrOvK+FqqOQXsLkey2nQBVRyRDk2kzsEc147MQZ3ez9s6LVxUSoRTapCGvOb8JlQM0EqHfN9Fse5s66dNKT7M4m8uoQbGCMjBnPFtr7epVs6DHOE3ED8fdoWoSdFLjNMx/jZURFWNiqJfNP33nVZOgu83xhqshE6Lq2rbd8pXTeqWoweKjZ+U0Jz5SUSgyL2a/E7J0Px6VoDKSL5nBENq+zVcNg6X+q2Cf65ImNXeOxeOD3esbwEoFZiTGCWKII4e4vmhU5zJbbhoqLfcXIAOpVLP3k7Kc8TGRytDJ5E+PV9N9qGZ/Oi6L9zBiA/z4oxqCBo6oyOJungRJ970+dKwv+92R1BRbyYdN2ICBuqjB4lylBGk721MqBY+iU4aEsGqW79u/Nu7Tga0CvVpRg06lmW24f1ThcX5tzcM+yVg1BnrOrY0jwRNbaok6NAKe/W6KA+sAUCMDLktqFi+mBbCN/0zcZVQmFdEnZF6nzwUeA+YgXskyKO7PMrh7ysef9oSuQg+xqL04YwkTG/OXa1nDtJZ1fG5ev2Hilj0Qc8usiEBpVnTo9iaUoiIAgaBnwJawtfYC5lQZ31hMF+bKo+oFhABTomJ94F71bctbKtO+7TqI51GV1WhQZEqTbBiObBjVkqJXtEmS+m5/jbDFRv7ZW2oTwqh8pZvaREUABlbj0fidbmKTlUYRgDJiZV630Nc8NRLUSdTlzF5RK4YW/cnAGzKbFU98yKE2sxKmQRHdCc+lWv6kR3LErK4vtur/s6K+3px+vwcb5Qcg5m4wZ2Qi2Rr6SWfN/UxAnQQh7TifgLHq+IOE+76OF2kdfwAC+8Ll0ajYaD9rGBuX2A82g7RG+wF6+kz7ICoOMEoJZ9i9BrG+WJJYf++nd5/0bnUIKOvaIyu7jQHULFAvg0ndds0uRqXIAt62FVlNAJVu0fmA1D4+bnQfWOnk3n0mAFVfX5J69xAvgplUgzsnxXXWbI+RxKRmJQpp6P3wk/pSqb18KNOYpe1t22xqxs+lK1wqJlUzxMp7gb7EpEK5GKdzLdYxc2vny12KMorSTntQJsetB37az81IwEoAkhS/fu6Wk5oECnIxfLhac0BGoQorrytcZspI0YBOKmspXe2gJ/28nwym2rXRpGhAR/NTXGaIyCikEcO0WHeyIzWwi5P5EHA3nCGjs28P7Zinw2sy6uUSWr1cMgwYyagsHNNJcwWaxsRk1eR8GXiwP0yWVM+fWH1VcXq6CowU8SdDiEep11koQ63PD8zNidFepEdWESW3eqW017RFVk184RkAI9Q/bAcl1d8ntPysgtF0xCGrNGxBRiZAbFfNk5r6maQx7zm7NKwiIv2eH8b7VoSz2IRUY5+Tmm1aNY4/2Tc18xP9/JmSsSwUrxeF4x6Wag8pe1I9fdFhf2zKMPAh1dNnkp90xa6dkabzZKjDtOV3WhhNqq3PC889HNc+kW9Qau+uFO4SqDyEkxKNdPW5RlIzPxG7qZET/gPVOHwurwB+HvbuvIdUO6mePpLWBzsbYDk/JFSFQ3IJpg9jzj20YvFA4D5e7qSg2uSXh1WwKiaaJ9jmB6eYPKl+HMfkJz60xXnylq6l7qzeOQBQJF6UkvawPcmDUAFlnA3imSyhCi6Ipu8qjBdRj/jJu1HjAUXrs2+jP4+hka2GackNUaXuyLKDKUXCsJkZIEXcyRsjo87Zse1ST/ZFmTRLfi+AJlLAQab1XH2vhxRpJ1PnykOSm7tRmQlp1L46RJV0+56M7pjSCovFkqpbjddaM0hP+JT6TnSnn6Qh1bEnc1y3x7uTIu2MEb01u1lDuhvmOX9QabITNZmFDdOY+Jss0uZaSJOKxwv5ZVLdeXgbChcnDnfu7SpHh0a9mru/JzVnmfAKfUSVm8t8xmgKl7JRfhTfeFVQ75nExFnLeD+EVA/2SUPRU3hy4rzVuE4Z11EGy+56M68bTanXaHrawrS+vHwjcTrwaFgjUAbW11eC8MwrQX2F8WpYFvOfwVxuR6Lak0dwa/BLg2fO+gGYr7X1jhFfeaQdzpd6km+dTLdnKMyFDCiFqrmeXzsqHX7X6+yosXO5H9xM+8HH6hfa62biR4zbTjJTG88ZeAbmssdl8WHqPWtpVr3wwAUpmk76kkf1Mjg+MN8IQ18nlW+zY3TXmzePEiYnpTpKBTvjLPndbVWhoHpKBfheHWJJsQ1Q5s4ZAay75QCkJnTSbRa33/Qs9YU+BwzzemXh645kUgydFLtvN2FS1CU/cxp08jlOTVXwvQTe3SJz0xnbAAtOj/tVyc/gw4JKTdK0zmKc9GkcHZCw3dENXd3cRMQKJixnF3H8xPw6UE6a3YAeu7FEspWrA+QwgVYq2D/6pJ8fkRfXYBtRVL4OIPrpmwh9b3n3Kh/0XuU7fUy6QU8qN+Ksj3gOZVPt/vDWrNwnTqZ5UVKg+LPjpEnxdlIwbQfla7heSqXaIm5OL3RGFcrLg4zthfrTbtOpJj3ivoLiBTfz5U6N3ozyQt/0wZCuHjvVlieFQ37yWtdUsE6xctKiPDqjZSeKUxM4UT1pcCnr4xQ3F2+VTW7P+TlTdTneihXTBU7NqUY8lKc4r+fBnSbomC0cjw84Op7HMrKqo0lKr5PZwz3/D1aTLjf7jgAA"
REL_SHA = "3296c826cf1e508e"
p = find('relectura_etiquetas.csv')
if p:
    raw = open(p, 'rb').read(); print('relectura desde dataset:', p)
else:
    raw = gzip.decompress(base64.b64decode(REL_B64)); print('relectura desde la copia incrustada')
print('sha256:', hashlib.sha256(raw).hexdigest()[:16], '| esperado (copia del 3-nov):', REL_SHA)
rel = pd.read_csv(io.BytesIO(raw), dtype={'micdas_original': str, 'micdas_adjudicado': str, 'micdas_modelos_seleccion': str, 'crop_id': str})
print(len(rel), 'filas |', rel.origen.value_counts().to_dict(), '|', rel.grupo.value_counts().to_dict())
print('relectura vacía:', rel.icdas_relectura.isna().sum(), '| cambios de mICDAS en el archivo:', int(rel.cambio_micdas.sum()))

In [ ]:
# ---- manifiesto v4 ----
zp = find('icdas_v4.zip')
if zp:                                             # el manifiesto dentro del zip de base-icdas-v4 es la fuente (igual que en pipeline-icdas-v4)
    z = zipfile.ZipFile(zp); name = [n for n in z.namelist() if n.endswith('manifest.csv')][0]
    man = pd.read_csv(z.open(name), dtype={'micdas': str, 'crop_id': str}); print('manifiesto:', zp, '→', name)
else:
    cands = [pd.read_csv(f, dtype={'micdas': str, 'crop_id': str}) for f in glob.glob(f'{IN}/**/manifest.csv', recursive=True)]
    man = [c for c in cands if {'crop_id', 'micdas', 'child_id'} <= set(c.columns)][0]
pp = find('manifest_privado_v4.csv')             # trae crop_id_v3 (= crop_id de v4) ↔ crop_id_anterior (icdas_NNNNN / cand_NNNNN), el ID que usó la relectura
if pp:
    priv = pd.read_csv(pp, dtype=str)[['crop_id_v3', 'crop_id_anterior']].drop_duplicates('crop_id_v3')
    man = man.merge(priv.rename(columns={'crop_id_v3': 'crop_id'}), on='crop_id', how='left')
    print('manifiesto privado:', pp, '| crop_id_anterior presente en', int(man.crop_id_anterior.notna().sum()), 'recortes')
print(len(man), 'recortes v4 | columnas:', list(man.columns))
print(man.micdas.value_counts().reindex(CLASSES).to_dict())

In [ ]:
# ---- cruce relectura ↔ manifiesto ----
ids = set(man.crop_id)
rel['en_v4'] = rel.crop_id.isin(ids); rel['crop_v4'] = np.where(rel.en_v4, rel.crop_id, None)
print('Por crop_id:', rel.groupby('origen').en_v4.agg(['sum', 'size']).to_dict('index'))
if (~rel.en_v4).any():
    # otra columna del manifiesto que guarde el ID antiguo (icdas_/cand_)
    falta = rel.loc[~rel.en_v4, 'crop_id']
    for col in man.columns:
        if col == 'crop_id' or pd.api.types.is_numeric_dtype(man[col]): continue
        mm = man[['crop_id', col]].dropna().astype({col: str}).drop_duplicates(col).set_index(col).crop_id
        sel = ~rel.en_v4 & rel.crop_id.isin(mm.index)
        if sel.any():
            rel.loc[sel, 'crop_v4'] = rel.loc[sel, 'crop_id'].map(mm); rel.loc[sel, 'en_v4'] = True
            print(f'  + {int(sel.sum())} calzados por la columna {col!r}')
    if (~rel.en_v4).any(): print('  (el CSV de relectura no trae foto ni caja, así que el cruce por ahmed_image + IoU no es posible aquí)')
print('No calzan:', int((~rel.en_v4).sum()), rel.loc[~rel.en_v4, 'crop_id'].head(10).tolist())
assert rel.crop_v4.dropna().is_unique, 'dos filas de relectura apuntan al mismo recorte v4'
mi = man.set_index('crop_id')
r4 = rel[rel.en_v4].copy(); r4['micdas_v4'] = mi.micdas.reindex(r4.crop_v4).values
r4['child_v4'] = mi.child_id.reindex(r4.crop_v4).values if 'child_id' in mi else None
r4['fold_v4'] = mi.fold.reindex(r4.crop_v4).values if 'fold' in mi else None
print('micdas v4 = micdas_adjudicado:', round((r4.micdas_v4 == r4.micdas_adjudicado).mean(), 4), f'({(r4.micdas_v4 == r4.micdas_adjudicado).sum()}/{len(r4)})')
print('micdas v4 = micdas_original:', round((r4.micdas_v4 == r4.micdas_original).mean(), 4))
print('child_id coincide:', (r4.child_v4 == r4.child_id).mean(), '| fold coincide:', (r4.fold_v4.astype(float) == r4.fold.astype(float)).mean())
dif = r4[r4.micdas_v4 != r4.micdas_adjudicado]
if len(dif): print('Casos v4 ≠ adjudicado:'); print(dif[['id', 'crop_id', 'micdas_original', 'micdas_adjudicado', 'micdas_v4', 'decision']].to_string())
print('Recortes cuya etiqueta cambia en el escenario 2:', int((r4.micdas_v4 != r4.micdas_original).sum()))
print(pd.crosstab(r4.micdas_v4, r4.micdas_original, rownames=['v4'], colnames=['original']))
r4[['id', 'crop_id', 'crop_v4', 'origen', 'grupo', 'micdas_original', 'micdas_adjudicado', 'micdas_v4']].to_csv(f'{W}/relectura_cruce_v4.csv', index=False)

In [ ]:
# ---- predicciones fuera de fold y etiquetas por escenario ----
oof = pd.read_csv(find('oof_predicciones.csv'), dtype={'crop_id': str})
if 'pred' not in oof:
    pc = [c for c in oof.columns if c.startswith('p_')]; oof['pred'] = oof[pc].values.argmax(1)
if oof.pred.dtype == object: oof['pred'] = oof.pred.map(CI)
MODELS = [m for m in NAMES if m in set(oof.model)] + sorted(set(oof.model) - set(NAMES))
print('modelos:', MODELS)
base = man[['crop_id', 'child_id', 'micdas']].copy(); base['y1'] = base.micdas.map(CI)
orig = r4.set_index('crop_v4').micdas_original.map(CI)
base['y2'] = base.y1; sel = base.crop_id.isin(orig.index); base.loc[sel, 'y2'] = base.loc[sel, 'crop_id'].map(orig).values
base['releido'] = sel
SCEN = {'1 v4': ('y1', np.ones(len(base), bool)), '2 originales': ('y2', np.ones(len(base), bool)), '3 sin releídos': ('y1', ~base.releido.values)}
for s, (yc, keep) in SCEN.items():
    print(f'{s:15s} n={keep.sum():5d} | niños={base.child_id[keep].nunique()} |', pd.Series(base[yc][keep]).map(dict(enumerate(CLASSES))).value_counts().reindex(CLASSES).to_dict())
P = {}
for m in MODELS:
    d = oof[oof.model == m].drop_duplicates('crop_id').set_index('crop_id').pred
    miss = base.crop_id[~base.crop_id.isin(d.index)]
    assert len(miss) == 0, f'{m}: {len(miss)} recortes sin predicción OOF'
    P[m] = d.reindex(base.crop_id).values.astype(int)

In [ ]:
# ---- métricas desde matrices de confusión por niño (bootstrap vectorizado) ----
kids = np.array(sorted(base.child_id.unique())); kix = pd.Series(np.arange(len(kids)), kids).reindex(base.child_id).values
def child_cm(y, p, keep):
    cm = np.zeros((len(kids), 4, 4)); np.add.at(cm, (kix[keep], y[keep], p[keep]), 1); return cm
def metrics(cm):                                   # cm: (..., 4, 4) filas = verdad, columnas = predicción
    tp = np.diagonal(cm, axis1=-2, axis2=-1); rs = cm.sum(-1); cs = cm.sum(-2); n = cm.sum((-2, -1))
    f1 = np.where(rs + cs > 0, 2 * tp / np.maximum(rs + cs, 1e-9), 0).mean(-1)
    w = np.subtract.outer(np.arange(4), np.arange(4)) ** 2 / 9.0
    exp = rs[..., :, None] * cs[..., None, :] / np.maximum(n, 1e-9)[..., None, None]
    kq = 1 - (w * cm).sum((-2, -1)) / np.maximum((w * exp).sum((-2, -1)), 1e-9)
    def sesp(c0):
        t = cm[..., c0:, :].sum(-2); f = cm[..., :c0, :].sum(-2)
        return t[..., c0:].sum(-1) / np.maximum(t.sum(-1), 1e-9), f[..., :c0].sum(-1) / np.maximum(f.sum(-1), 1e-9)
    se_bc, sp_bc = sesp(2); se_any, sp_any = sesp(1)
    recC = cm[..., 3, 3] / np.maximum(rs[..., 3], 1e-9)
    return dict(macro_f1=f1, kappa_q=kq, se_BC=se_bc, sp_BC=sp_bc, se_caries=se_any, sp_caries=sp_any, recall_C=recC, n=n)
rng = np.random.default_rng(SEED)
W_BOOT = np.stack([np.bincount(rng.integers(0, len(kids), len(kids)), minlength=len(kids)) for _ in range(NB)]).astype(float)   # NB × niños
CM, PT, BS = {}, {}, {}
for s, (yc, keep) in SCEN.items():
    y = base[yc].values.astype(int)
    for m in MODELS:
        cm = child_cm(y, P[m], keep); CM[s, m] = cm
        PT[s, m] = {k: float(v) for k, v in metrics(cm.sum(0)).items()}
        BS[s, m] = metrics(np.einsum('bk,kij->bij', W_BOOT, cm))
from sklearn.metrics import f1_score, cohen_kappa_score
_y, _k = base['y1'].values.astype(int), SCEN['1 v4'][1]
for m in MODELS:
    a = PT['1 v4', m]; assert abs(a['macro_f1'] - f1_score(_y, P[m], average='macro', labels=[0, 1, 2, 3])) < 1e-9 and abs(a['kappa_q'] - cohen_kappa_score(_y, P[m], weights='quadratic')) < 1e-9
print('control: macro-F1 y κ desde matrices de confusión = sklearn ✓')

In [ ]:
# ---- tabla por modelo y escenario ----
rows = []
for s in SCEN:
    for m in MODELS:
        pt, bs = PT[s, m], BS[s, m]
        r = dict(escenario=s, modelo=NAMES.get(m, m), n=int(pt['n']))
        for k in ('macro_f1', 'kappa_q', 'se_BC', 'sp_BC', 'se_caries', 'sp_caries', 'recall_C'):
            r[k] = pt[k]; r[k + '_lo'] = np.quantile(bs[k], .025); r[k + '_hi'] = np.quantile(bs[k], .975)
        if s != '1 v4':
            d = bs['macro_f1'] - BS['1 v4', m]['macro_f1']
            r['d_macro_f1_vs_v4'] = pt['macro_f1'] - PT['1 v4', m]['macro_f1']; r['d_lo'] = np.quantile(d, .025); r['d_hi'] = np.quantile(d, .975)
        rows.append(r)
T = pd.DataFrame(rows); T.to_csv(f'{W}/sensibilidad_relectura_por_modelo.csv', index=False)
show = T.copy()
show['macro-F1 (IC 95 %)'] = show.apply(lambda r: f"{r.macro_f1:.3f} ({r.macro_f1_lo:.3f}–{r.macro_f1_hi:.3f})", axis=1)
show['Δ vs v4 (IC 95 %)'] = show.apply(lambda r: '' if pd.isna(r.get('d_macro_f1_vs_v4')) else f"{r.d_macro_f1_vs_v4:+.3f} ({r.d_lo:+.3f} a {r.d_hi:+.3f})", axis=1)
print(show[['escenario', 'modelo', 'n', 'macro-F1 (IC 95 %)', 'Δ vs v4 (IC 95 %)', 'kappa_q', 'se_BC', 'sp_BC', 'se_caries', 'sp_caries', 'recall_C']].round(3).to_string(index=False))

In [ ]:
# ---- orden de los modelos y Swin-T vs cada uno (bootstrap pareado por niño, Holm) ----
def holm(p):
    p = np.asarray(p); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for r, i in enumerate(o): run = max(run, (len(p) - r) * p[i]); adj[i] = min(1, run)
    return adj
H = []
for s in SCEN:
    order = sorted(MODELS, key=lambda m: -PT[s, m]['macro_f1'])
    print(f'\nEscenario {s} — orden por macro-F1:', ' > '.join(f"{NAMES.get(m, m)} {PT[s, m]['macro_f1']:.3f}" for m in order))
    others = [m for m in MODELS if m != REF]; ds, ps = [], []
    for m in others:
        d = BS[s, REF]['macro_f1'] - BS[s, m]['macro_f1']; ds.append(PT[s, REF]['macro_f1'] - PT[s, m]['macro_f1'])
        ps.append(min(1, 2 * min((d <= 0).mean(), (d >= 0).mean())))
    pa = holm(ps)
    for m, dd, p_, a in zip(others, ds, ps, pa):
        H.append(dict(escenario=s, comparacion=f'Swin-T − {NAMES.get(m, m)}', diferencia=dd, p=p_, p_holm=a, significativo=a < 0.05))
        print(f'   Swin-T − {NAMES.get(m, m):13s} {dd:+.3f}  p_Holm {a:.3f}{"  *" if a < 0.05 else ""}')
pd.DataFrame(H).to_csv(f'{W}/sensibilidad_relectura_holm.csv', index=False)

In [ ]:
# ---- concordancia en los controles (relectura ciega vs etiqueta original) ----
from sklearn.metrics import cohen_kappa_score
to_m = lambda x: np.select([x >= 5, x >= 3, x >= 1], [3, 2, 1], 0)
C = []
for g, s in rel.dropna(subset=['icdas_relectura']).groupby('grupo'):
    a, b = s.icdas_original.astype(int).values, s.icdas_relectura.astype(int).values
    C.append(dict(grupo=g, n=len(s), acuerdo_exacto_icdas=(a == b).mean(), kappa_lineal_icdas=cohen_kappa_score(a, b, weights='linear'),
                  kappa_micdas=cohen_kappa_score(to_m(a), to_m(b)), kappa_lineal_micdas=cohen_kappa_score(to_m(a), to_m(b), weights='linear')))
C = pd.DataFrame(C); C.to_csv(f'{W}/relectura_concordancia.csv', index=False)
print(C.round(3).to_string(index=False)); print('Esperado en controles: κ lineal ICDAS 0,74 | κ mICDAS 0,49 | acuerdo exacto 0,55')

In [ ]:
# ---- resumen para traer de vuelta ----
print('=== Cruce con el manifiesto v4 ===')
print(f"releídos en v4: {int(rel.en_v4.sum())}/{len(rel)} (base {int(rel[rel.origen=='base'].en_v4.sum())}/{(rel.origen=='base').sum()}, "
      f"minado2 {int(rel[rel.origen=='minado2'].en_v4.sum())}/{(rel.origen=='minado2').sum()}) | v4 = adjudicado en {(r4.micdas_v4 == r4.micdas_adjudicado).sum()}/{len(r4)} "
      f"| etiquetas que cambian en el escenario 2: {int((r4.micdas_v4 != r4.micdas_original).sum())}")
print('=== Swin-T por escenario ===')
for s in SCEN:
    r = T[(T.escenario == s) & (T.modelo == 'Swin-T')].iloc[0]
    extra = '' if s == '1 v4' else f" | Δ vs v4 {r.d_macro_f1_vs_v4:+.3f} ({r.d_lo:+.3f} a {r.d_hi:+.3f})"
    print(f"{s:15s} n={r.n:5d} | macro-F1 {r.macro_f1:.3f} ({r.macro_f1_lo:.3f}–{r.macro_f1_hi:.3f}) | κq {r.kappa_q:.3f} | Se/Sp B+C {r.se_BC:.3f}/{r.sp_BC:.3f} | recall C {r.recall_C:.3f}{extra}")
sig = pd.DataFrame(H); sig = sig[sig.significativo]
print('=== Significativamente peores que Swin-T (Holm) ===')
for s in SCEN: print(f"{s:15s}", ', '.join(sig[sig.escenario == s].comparacion.str.replace('Swin-T − ', '')) or 'ninguno')
print(sorted(os.listdir(W)))